# Notebook Overview — Basic Fine-Tuning

## Purpose

This notebook performs **controlled hyperparameter tuning** of the baseline classifier to improve performance while preserving the original Digital Image Processing (DIP) feature-based approach.

The goal is to demonstrate that performance gains can be achieved through modest model refinement without altering the feature engineering methodology.

---

## Inputs

The notebook expects the following:

* Training feature vectors from the GitHub repository:

  * `metadata/vectors/train_feature_vectors.csv`

* Test feature vectors from the GitHub repository:

  * `metadata/vectors/test_feature_vectors.csv`

* Project configuration file:

  * `src/project_config.py`

---

## Assumptions

* Each feature vector CSV contains the columns:

  * `filename`
  * `class_label`
  * `source_dataset`
  * `subset`

* The remaining columns represent the engineered DIP feature vector

* The feature vector data has already been created by earlier notebooks in the pipeline

* The training and test sets are already defined and must remain separate

* Cross-validation is performed using the training dataset only

* The test set is used only once for final evaluation of the selected tuned model

* This notebook focuses on **controlled tuning** of the classifier and does not alter the feature extraction process

* All outputs are written to **local runtime storage only**

* No files are written back to Google Drive

---

## What This Notebook Does

### Cell 1 — Environment Setup

* Clone the GitHub repository into the local runtime  
* Import project configuration  
* Verify required input files are available  

---

### Cell 2 — Load Feature Vector Data

* Load the training and test feature vector CSV files  
* Display dataset shapes and sample rows  
* Verify expected metadata columns are present  

---

### Cell 3 — Prepare Features and Labels

* Separate metadata columns from feature columns  
* Create feature matrices (`X_train`, `X_test`)  
* Create label vectors (`y_train`, `y_test`)  
* Verify feature dimensionality and class distribution  

---

### Cell 4 — Normalize Features

* Fit a scaler using the training feature matrix  
* Transform both training and test feature matrices  
* Preserve train/test separation during normalization  
* Save the fitted scaler for reuse  

---

### Cell 5 — Define Candidate Model Configurations

* Define a small set of candidate MLP configurations  
* Vary selected hyperparameters such as:

  * Hidden layer structure  
  * Regularization strength (`alpha`)  
  * Learning rate (optional)  

---

### Cell 6 — Perform Cross-Validation on Training Data

* Evaluate candidate configurations using stratified k-fold cross-validation  
* Compute evaluation metrics:

  * Accuracy  
  * Precision  
  * Recall  
  * F1-score  
  * ROC AUC  

* Model selection is based primarily on **ROC AUC**, with **F1-score** used as a secondary metric  

---

### Cell 7 — Select Best Configuration

* Compare candidate models using cross-validation results  
* Select the best-performing configuration  
* The selected model may be a lower-complexity architecture, reflecting the structured nature of the engineered feature space  

---

### Cell 8 — Train Final Tuned Model

* Retrain the selected model using the full normalized training dataset  
* Confirm convergence behavior and training stability  

---

### Cell 9 — Evaluate Tuned Model on Test Set

* Generate predictions on the independent test set  
* Compute evaluation metrics:

  * Accuracy  
  * Precision  
  * Recall  
  * F1-score  
  * ROC AUC  

* Compute confusion matrix and classification report  

---

### Cell 10 — Display Results

* Present cross-validation summaries and final test metrics  
* Display confusion matrix and ROC curve visualizations  

---

### Cell 11 — Save Results

* Save tuning results, best model configuration, and final evaluation outputs  
* Verify file creation  

---

### Cell 12 — Final Summary

* Display fine-tuning summary  
* Compare tuned performance against the baseline  
* Provide guidance for next steps in the pipeline  

---

## Outputs

The following outputs are generated in the **local runtime**:

* Cross-validation performance summaries  
* Final tuned model test-set metrics  
* Confusion matrix  
* ROC curve  

Saved output files:

* `metadata/results/fine_tuning_results.csv`  
* `metadata/results/best_model_config.json`  
* `metadata/results/tuned_model_results.csv`  
* `metadata/models/scaler.pkl`  

These files are not written back to Google Drive and will be lost when the runtime ends unless explicitly preserved.

---

## Expected Data Scope

* Training feature vectors: **14,400 rows**  
* Test feature vectors: **3,600 rows**

Each row corresponds to one image and includes metadata plus the engineered DIP feature vector.

---

## Notes

* This notebook focuses on **controlled improvement** of the baseline classifier  
* The primary goal is to determine whether modest tuning improves performance beyond the baseline established in Notebook 10  
* Cross-validation is applied only to the training dataset  
* The test set remains independent and is used only for final tuned-model evaluation  
* The Multi-Layer Perceptron (MLP) remains the primary classifier used in this notebook  
* Hyperparameter exploration is intentionally limited to maintain interpretability and reduce overfitting risk  
* The selected tuned model is expected to converge within the iteration limit without warnings, indicating stable training behavior  
* Results from this notebook are intended to be compared directly with the baseline results from Notebook 10  

---

**Next step:**  
Use the results from this notebook together with those from **10_Basic_Testing.ipynb** to analyze performance improvements and support final reporting.

In [ ]:
# ============================================
# Cell 1: Startup (Environment + Verification)
# ============================================

import os
import sys
from pathlib import Path

# -------------------------------------------------
# Clone repo into Colab runtime (if not already present)
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Make src/ importable
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import project configuration
# -------------------------------------------------
from project_config import *

# -------------------------------------------------
# Define paths
# -------------------------------------------------
METADATA_ROOT = REPO_DIR / "metadata"
VECTORS_DIR = METADATA_ROOT / "vectors"
RESULTS_DIR = METADATA_ROOT / "results"
MODELS_DIR = METADATA_ROOT / "models"

TRAIN_FEATURE_FILE = VECTORS_DIR / "train_feature_vectors.csv"
TEST_FEATURE_FILE  = VECTORS_DIR / "test_feature_vectors.csv"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_RESULTS_FILE = RESULTS_DIR / "fine_tuning_results.csv"
BEST_MODEL_CONFIG_FILE = RESULTS_DIR / "best_model_config.json"

# -------------------------------------------------
# Verify required input files
# -------------------------------------------------
print("Verifying required input files...\n")

missing_files = []

for file_path in [TRAIN_FEATURE_FILE, TEST_FEATURE_FILE]:
    if not file_path.exists():
        missing_files.append(file_path.name)

if missing_files:
    raise FileNotFoundError(
        f"Missing required feature vector files in metadata/vectors/: {missing_files}"
    )

print("All required input files are present.")
print(f"Training features: {TRAIN_FEATURE_FILE}")
print(f"Test features:     {TEST_FEATURE_FILE}")
print(f"Results file:      {OUTPUT_RESULTS_FILE}")

print("\nStartup complete.")



In [ ]:
# ============================================
# Cell 2: Load Feature Vector Data
# ============================================

import pandas as pd

# -------------------------------------------------
# Load training and test feature vectors
# -------------------------------------------------
train_df = pd.read_csv(TRAIN_FEATURE_FILE)
test_df  = pd.read_csv(TEST_FEATURE_FILE)

print("Feature vector files loaded successfully.\n")

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape:     {test_df.shape}")

# -------------------------------------------------
# Verify required columns
# -------------------------------------------------
required_columns = METADATA_COLUMNS

missing_train = [col for col in required_columns if col not in train_df.columns]
missing_test  = [col for col in required_columns if col not in test_df.columns]

if missing_train:
    raise ValueError(f"Training file missing required columns: {missing_train}")

if missing_test:
    raise ValueError(f"Test file missing required columns: {missing_test}")

print("\nRequired metadata columns verified.")

# -------------------------------------------------
# Verify subset labels
# -------------------------------------------------
train_subsets = train_df["subset"].unique()
test_subsets  = test_df["subset"].unique()

if list(train_subsets) != [TRAIN_SUBSET]:
    raise ValueError(f"Unexpected subset values in training data: {train_subsets}")

if list(test_subsets) != [TEST_SUBSET]:
    raise ValueError(f"Unexpected subset values in test data: {test_subsets}")

print("Subset labels verified.")

# -------------------------------------------------
# Check for missing values
# -------------------------------------------------
if train_df.isnull().values.any():
    raise ValueError("Training data contains missing values.")

if test_df.isnull().values.any():
    raise ValueError("Test data contains missing values.")

print("No missing values detected.")

# -------------------------------------------------
# Display class distribution
# -------------------------------------------------
print("\nTraining class distribution:")
print(train_df["class_label"].value_counts())

print("\nTest class distribution:")
print(test_df["class_label"].value_counts())

# -------------------------------------------------
# Display sample rows
# -------------------------------------------------
print("\nSample training rows:")
display(train_df.head())

print("\nSample test rows:")
display(test_df.head())



In [ ]:
# ============================================
# Cell 3: Prepare Features and Labels
# ============================================

# -------------------------------------------------
# Use config-defined metadata columns
# -------------------------------------------------
metadata_columns = METADATA_COLUMNS

feature_columns = [col for col in train_df.columns if col not in metadata_columns]

print(f"Number of feature columns: {len(feature_columns)}")
print(f"Expected number of features: {NUM_FEATURES}")

# -------------------------------------------------
# Validate feature count
# -------------------------------------------------
if len(feature_columns) != NUM_FEATURES:
    raise ValueError(
        f"Feature count mismatch. Expected {NUM_FEATURES}, found {len(feature_columns)}"
    )

# -------------------------------------------------
# Create feature matrices
# -------------------------------------------------
X_train = train_df[feature_columns].values
X_test  = test_df[feature_columns].values

# -------------------------------------------------
# Create label vectors
# -------------------------------------------------
y_train = train_df["class_label"].values
y_test  = test_df["class_label"].values

# -------------------------------------------------
# Verify feature consistency
# -------------------------------------------------
if X_train.shape[1] != X_test.shape[1]:
    raise ValueError(
        f"Feature dimension mismatch: train={X_train.shape[1]}, test={X_test.shape[1]}"
    )

print("\nFeature matrices created.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

# -------------------------------------------------
# Verify class labels
# -------------------------------------------------
unique_labels = sorted(set(y_train) | set(y_test))

print(f"\nUnique class labels: {unique_labels}")

if set(unique_labels) != set(VALID_LABELS):
    raise ValueError(
        f"Unexpected class labels. Expected {VALID_LABELS}, found {unique_labels}"
    )

print("Class labels verified.")

# -------------------------------------------------
# Display sample feature row
# -------------------------------------------------
print("\nSample feature vector (first row):")
print(X_train[0])



In [ ]:
# ============================================
# Cell 4: Normalize Features
# ============================================

from sklearn.preprocessing import StandardScaler
import joblib

# -------------------------------------------------
# Define local scaler output path
# -------------------------------------------------
SCALER_FILE = MODELS_DIR / SCALER_FILENAME

# -------------------------------------------------
# Fit scaler on training data only
# -------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Feature normalization complete.\n")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")

# -------------------------------------------------
# Save scaler to local runtime
# -------------------------------------------------
joblib.dump(scaler, SCALER_FILE)

if not SCALER_FILE.exists():
    raise FileNotFoundError(f"Scaler file was not created: {SCALER_FILE}")

print(f"Scaler saved to: {SCALER_FILE}")

# -------------------------------------------------
# Sanity check on normalized training data
# -------------------------------------------------
train_mean = X_train_scaled.mean(axis=0)
train_std = X_train_scaled.std(axis=0)

print("\nNormalization sanity check:")
print(f"Mean of first 5 scaled features: {train_mean[:5]}")
print(f"Std of first 5 scaled features:  {train_std[:5]}")



In [ ]:
# ============================================
# Cell 5: Define Candidate Model Configurations
# ============================================

# -------------------------------------------------
# Define a small set of candidate MLP configurations
# -------------------------------------------------
candidate_configs = [
    {
        "model_name": "MLP_Small_alpha_0.0001",
        "hidden_layer_sizes": (32,),
        "alpha": 0.0001,
        "learning_rate_init": 0.001,
        "max_iter": 300,
    },
    {
        "model_name": "MLP_Medium_alpha_0.0001",
        "hidden_layer_sizes": (64, 32),
        "alpha": 0.0001,
        "learning_rate_init": 0.001,
        "max_iter": 300,
    },
    {
        "model_name": "MLP_Large_alpha_0.0001",
        "hidden_layer_sizes": (128, 64, 32),
        "alpha": 0.0001,
        "learning_rate_init": 0.001,
        "max_iter": 300,
    },
    {
        "model_name": "MLP_Medium_alpha_0.001",
        "hidden_layer_sizes": (64, 32),
        "alpha": 0.001,
        "learning_rate_init": 0.001,
        "max_iter": 300,
    },
    {
        "model_name": "MLP_Medium_alpha_0.00001",
        "hidden_layer_sizes": (64, 32),
        "alpha": 0.00001,
        "learning_rate_init": 0.001,
        "max_iter": 300,
    },
]

print("Candidate model configurations defined.\n")
print(f"Number of candidate configurations: {len(candidate_configs)}\n")

for i, config in enumerate(candidate_configs, start=1):
    print(f"Configuration {i}:")
    print(f"  model_name         : {config['model_name']}")
    print(f"  hidden_layer_sizes : {config['hidden_layer_sizes']}")
    print(f"  alpha              : {config['alpha']}")
    print(f"  learning_rate_init : {config['learning_rate_init']}")
    print(f"  max_iter           : {config['max_iter']}")
    print()



In [ ]:
# ============================================
# Cell 6: Perform Cross-Validation on Training Data
# ============================================

import time
import warnings
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

# -------------------------------------------------
# Define cross-validation strategy
# -------------------------------------------------
cv = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=CV_SHUFFLE,
    random_state=CV_RANDOM_STATE
)

# -------------------------------------------------
# Define scoring metrics
# -------------------------------------------------
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, pos_label=AI_LABEL),
    "recall": make_scorer(recall_score, pos_label=AI_LABEL),
    "f1": make_scorer(f1_score, pos_label=AI_LABEL),
    "roc_auc": "roc_auc",
}

# -------------------------------------------------
# Evaluate candidate configurations
# -------------------------------------------------
cv_results = []

print("Beginning cross-validation on training data...\n")

for config in candidate_configs:
    print(f"Evaluating: {config['model_name']}")

    model = MLPClassifier(
        hidden_layer_sizes=config["hidden_layer_sizes"],
        activation="relu",
        solver="adam",
        alpha=config["alpha"],
        batch_size=32,
        learning_rate_init=config["learning_rate_init"],
        max_iter=config["max_iter"],
        random_state=RANDOM_SEED,
        early_stopping=False,
    )

    start_time = time.time()

    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter("always", ConvergenceWarning)

        scores = cross_validate(
            model,
            X_train_scaled,
            y_train,
            cv=cv,
            scoring=scoring,
            return_train_score=False,
            n_jobs=-1,
        )

    elapsed_time = time.time() - start_time

    convergence_warning_count = sum(
        1 for w in caught_warnings if issubclass(w.category, ConvergenceWarning)
    )

    result = {
        "model_name": config["model_name"],
        "hidden_layer_sizes": str(config["hidden_layer_sizes"]),
        "alpha": config["alpha"],
        "learning_rate_init": config["learning_rate_init"],
        "max_iter": config["max_iter"],
        "cv_accuracy_mean": scores["test_accuracy"].mean(),
        "cv_accuracy_std": scores["test_accuracy"].std(),
        "cv_precision_mean": scores["test_precision"].mean(),
        "cv_precision_std": scores["test_precision"].std(),
        "cv_recall_mean": scores["test_recall"].mean(),
        "cv_recall_std": scores["test_recall"].std(),
        "cv_f1_mean": scores["test_f1"].mean(),
        "cv_f1_std": scores["test_f1"].std(),
        "cv_roc_auc_mean": scores["test_roc_auc"].mean(),
        "cv_roc_auc_std": scores["test_roc_auc"].std(),
        "fit_time_mean": scores["fit_time"].mean(),
        "score_time_mean": scores["score_time"].mean(),
        "elapsed_time_sec": elapsed_time,
        "convergence_warning_count": convergence_warning_count,
    }

    cv_results.append(result)
    print(
        f"  ROC AUC: {result['cv_roc_auc_mean']:.4f} | "
        f"F1: {result['cv_f1_mean']:.4f} | "
        f"Accuracy: {result['cv_accuracy_mean']:.4f}"
    )
    print()

# -------------------------------------------------
# Create results dataframe
# -------------------------------------------------
cv_results_df = pd.DataFrame(cv_results)

# -------------------------------------------------
# Sort by primary metric
# -------------------------------------------------
cv_results_df = cv_results_df.sort_values(
    by=["cv_roc_auc_mean", "cv_f1_mean"],
    ascending=False
).reset_index(drop=True)

print("Cross-validation complete.\n")
print("Top configurations:")
display(cv_results_df.head())


In [ ]:
# ============================================
# Cell 7: Select Best Configuration
# ============================================

import json

# -------------------------------------------------
# Confirm results are available
# -------------------------------------------------
if cv_results_df.empty:
    raise ValueError("Cross-validation results are empty. Run Cell 6 first.")

# -------------------------------------------------
# Select best configuration
# -------------------------------------------------
best_row = cv_results_df.iloc[0]

best_config = {
    "model_name": best_row["model_name"],
    "hidden_layer_sizes": best_row["hidden_layer_sizes"],
    "alpha": float(best_row["alpha"]),
    "learning_rate_init": float(best_row["learning_rate_init"]),
    "max_iter": int(best_row["max_iter"]),
    "cv_accuracy_mean": float(best_row["cv_accuracy_mean"]),
    "cv_accuracy_std": float(best_row["cv_accuracy_std"]),
    "cv_precision_mean": float(best_row["cv_precision_mean"]),
    "cv_precision_std": float(best_row["cv_precision_std"]),
    "cv_recall_mean": float(best_row["cv_recall_mean"]),
    "cv_recall_std": float(best_row["cv_recall_std"]),
    "cv_f1_mean": float(best_row["cv_f1_mean"]),
    "cv_f1_std": float(best_row["cv_f1_std"]),
    "cv_roc_auc_mean": float(best_row["cv_roc_auc_mean"]),
    "cv_roc_auc_std": float(best_row["cv_roc_auc_std"]),
    "convergence_warning_count": int(best_row["convergence_warning_count"]),
}

print("Best configuration selected.\n")

print(f"Model name         : {best_config['model_name']}")
print(f"Hidden layers      : {best_config['hidden_layer_sizes']}")
print(f"Alpha              : {best_config['alpha']}")
print(f"Learning rate      : {best_config['learning_rate_init']}")
print(f"Max iterations     : {best_config['max_iter']}")

print("\nCross-validation performance:")
print(f"Accuracy           : {best_config['cv_accuracy_mean']:.4f} ± {best_config['cv_accuracy_std']:.4f}")
print(f"Precision          : {best_config['cv_precision_mean']:.4f} ± {best_config['cv_precision_std']:.4f}")
print(f"Recall             : {best_config['cv_recall_mean']:.4f} ± {best_config['cv_recall_std']:.4f}")
print(f"F1-score           : {best_config['cv_f1_mean']:.4f} ± {best_config['cv_f1_std']:.4f}")
print(f"ROC AUC            : {best_config['cv_roc_auc_mean']:.4f} ± {best_config['cv_roc_auc_std']:.4f}")

print(f"\nConvergence warnings: {best_config['convergence_warning_count']}")

# -------------------------------------------------
# Optional: save best configuration immediately
# -------------------------------------------------
with open(BEST_MODEL_CONFIG_FILE, "w") as f:
    json.dump(best_config, f, indent=4)

print(f"\nBest configuration saved to: {BEST_MODEL_CONFIG_FILE}")



In [ ]:
# ============================================
# Cell 8: Train Final Tuned Model
# ============================================

import time
import warnings
from sklearn.neural_network import MLPClassifier
from sklearn.exceptions import ConvergenceWarning

# -------------------------------------------------
# Recover the selected candidate configuration
# -------------------------------------------------
selected_candidate = None

for config in candidate_configs:
    if config["model_name"] == best_config["model_name"]:
        selected_candidate = config
        break

if selected_candidate is None:
    raise ValueError(
        f"Could not locate selected configuration: {best_config['model_name']}"
    )

print("Training final tuned model...\n")
print(f"Selected model: {selected_candidate['model_name']}")

# -------------------------------------------------
# Define final tuned model
# -------------------------------------------------
final_mlp = MLPClassifier(
    hidden_layer_sizes=selected_candidate["hidden_layer_sizes"],
    activation="relu",
    solver="adam",
    alpha=selected_candidate["alpha"],
    batch_size=32,
    learning_rate_init=selected_candidate["learning_rate_init"],
    max_iter=selected_candidate["max_iter"],
    random_state=RANDOM_SEED,
    early_stopping=False,
)

# -------------------------------------------------
# Train final tuned model
# -------------------------------------------------
start_time = time.time()

with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always", ConvergenceWarning)
    final_mlp.fit(X_train_scaled, y_train)

end_time = time.time()
final_training_time = end_time - start_time

print("\nTraining complete.")
print(f"Training time: {final_training_time:.2f} seconds")
print(f"Number of iterations completed: {final_mlp.n_iter_}")
print(f"Final training loss: {final_mlp.loss_:.6f}")

# -------------------------------------------------
# Report convergence information
# -------------------------------------------------
convergence_warnings = [
    w for w in caught_warnings
    if issubclass(w.category, ConvergenceWarning)
]

if convergence_warnings:
    print("\nNote: A convergence warning was detected.")
    print("The optimizer reached the maximum number of iterations before full convergence.")
else:
    print("\nModel converged without warnings.")



In [ ]:
# ============================================
# Cell 9: Evaluate Tuned Model on Test Set
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# -------------------------------------------------
# Generate predictions
# -------------------------------------------------
y_pred = final_mlp.predict(X_test_scaled)

# Use correct probability column for AI class
ai_index = list(final_mlp.classes_).index(AI_LABEL)
y_prob = final_mlp.predict_proba(X_test_scaled)[:, ai_index]

# -------------------------------------------------
# Convert labels to binary for ROC AUC
# -------------------------------------------------
y_test_binary = (y_test == AI_LABEL).astype(int)

# -------------------------------------------------
# Compute evaluation metrics
# -------------------------------------------------
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=AI_LABEL)
recall = recall_score(y_test, y_pred, pos_label=AI_LABEL)
f1 = f1_score(y_test, y_pred, pos_label=AI_LABEL)
roc_auc = roc_auc_score(y_test_binary, y_prob)

cm = confusion_matrix(y_test, y_pred, labels=[REAL_LABEL, AI_LABEL])

# -------------------------------------------------
# Display results
# -------------------------------------------------
print("Tuned model test-set evaluation complete.\n")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix (rows=true, cols=pred):")
print(cm)

# -------------------------------------------------
# Human-readable confusion matrix summary
# -------------------------------------------------
rl_correct = cm[0, 0]
rl_misclassified = cm[0, 1]

ai_correct = cm[1, 1]
ai_misclassified = cm[1, 0]

print("\nConfusion Matrix Summary:")
print(f"Real ({REAL_LABEL}): {rl_correct} correct, {rl_misclassified} misclassified")
print(f"AI   ({AI_LABEL}): {ai_correct} correct, {ai_misclassified} misclassified")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))



In [ ]:
# ============================================
# Cell 10: Display Results
# ============================================

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

# -------------------------------------------------
# Cross-validation summary (sorted best first)
# -------------------------------------------------
print("Cross-validation summary (top configurations):")
display(
    cv_results_df[
        [
            "model_name",
            "hidden_layer_sizes",
            "alpha",
            "cv_accuracy_mean",
            "cv_f1_mean",
            "cv_roc_auc_mean",
            "convergence_warning_count",
        ]
    ].head()
)

# -------------------------------------------------
# Final tuned model metric summary
# -------------------------------------------------
tuned_results_summary_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC AUC"],
    "Value": [accuracy, precision, recall, f1, roc_auc]
})

print("\nFinal tuned-model test-set summary:")
display(tuned_results_summary_df)

# -------------------------------------------------
# Confusion matrix display
# -------------------------------------------------
disp_cm = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Real", "AI"]
)

fig, ax = plt.subplots(figsize=(6, 6))
disp_cm.plot(cmap="Blues", values_format="d", ax=ax)
ax.set_title("Tuned MLP Confusion Matrix (Real vs AI)")
ax.grid(False)
plt.show()

# -------------------------------------------------
# ROC curve display
# -------------------------------------------------
RocCurveDisplay.from_predictions(
    y_test_binary,
    y_prob,
    name="Tuned MLP"
)

plt.title("Tuned MLP ROC Curve")
plt.grid(True)
plt.show()



In [ ]:
# ============================================
# Cell 11: Save Results
# ============================================

import pandas as pd
import json

# -------------------------------------------------
# Save full cross-validation results
# -------------------------------------------------
cv_results_df.to_csv(OUTPUT_RESULTS_FILE, index=False)

if not OUTPUT_RESULTS_FILE.exists():
    raise FileNotFoundError(f"Results file was not created: {OUTPUT_RESULTS_FILE}")

print(f"Cross-validation results saved to: {OUTPUT_RESULTS_FILE}")

# -------------------------------------------------
# Save best model configuration
# -------------------------------------------------
with open(BEST_MODEL_CONFIG_FILE, "w") as f:
    json.dump(best_config, f, indent=4)

print(f"Best model configuration saved to: {BEST_MODEL_CONFIG_FILE}")

# -------------------------------------------------
# Save final tuned model metrics
# -------------------------------------------------
final_results_dict = {
    "model": best_config["model_name"],
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "roc_auc": roc_auc,
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "num_features": X_train.shape[1],
    "training_iterations": final_mlp.n_iter_,
    "final_loss": final_mlp.loss_,
}

FINAL_RESULTS_FILE = RESULTS_DIR / "tuned_model_results.csv"

final_results_df = pd.DataFrame([final_results_dict])
final_results_df.to_csv(FINAL_RESULTS_FILE, index=False)

if not FINAL_RESULTS_FILE.exists():
    raise FileNotFoundError(f"Tuned results file was not created: {FINAL_RESULTS_FILE}")

print(f"Final tuned model results saved to: {FINAL_RESULTS_FILE}")

print("\nCell complete.")



In [ ]:
# ============================================
# Cell 12: Final Summary and Comparison to Baseline
# ============================================

from IPython.display import display, HTML
import pandas as pd

print("Basic fine-tuning completed successfully.\n")

print("Best Tuned Model Summary:")
print(f"Model name         : {best_config['model_name']}")
print(f"Hidden layers      : {selected_candidate['hidden_layer_sizes']}")
print(f"Alpha              : {selected_candidate['alpha']}")
print(f"Learning rate      : {selected_candidate['learning_rate_init']}")
print(f"Training samples   : {len(X_train)}")
print(f"Test samples       : {len(X_test)}")
print(f"Feature count      : {X_train.shape[1]}")
print(f"Training iterations: {final_mlp.n_iter_}")
print(f"Final training loss: {final_mlp.loss_:.6f}")

print("\nTuned Test Metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix Summary:")
print(f"Real ({REAL_LABEL}): {cm[0,0]} correct, {cm[0,1]} misclassified")
print(f"AI   ({AI_LABEL}): {cm[1,1]} correct, {cm[1,0]} misclassified")

# -------------------------------------------------
# Compare against baseline results if available
# -------------------------------------------------
BASELINE_RESULTS_FILE = RESULTS_DIR / "basic_testing_results.csv"

if BASELINE_RESULTS_FILE.exists():
    baseline_df = pd.read_csv(BASELINE_RESULTS_FILE)

    if len(baseline_df) > 0:
        baseline_row = baseline_df.iloc[0]

        comparison_df = pd.DataFrame({
            "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC AUC"],
            "Baseline": [
                baseline_row["accuracy"],
                baseline_row["precision"],
                baseline_row["recall"],
                baseline_row["f1_score"],
                baseline_row["roc_auc"],
            ],
            "Tuned": [
                accuracy,
                precision,
                recall,
                f1,
                roc_auc,
            ],
        })

        comparison_df["Absolute Change"] = comparison_df["Tuned"] - comparison_df["Baseline"]

        print("\nBaseline vs Tuned Comparison:")
        display(comparison_df)
    else:
        print("\nBaseline results file exists but is empty. Skipping comparison.")
else:
    print("\nBaseline results file not found. Skipping baseline comparison.")

# -------------------------------------------------
# Final next-step message
# -------------------------------------------------
message = """
<b>Fine-Tuning Complete:</b><br>
The tuned MLP model has been selected using cross-validation on the training data and evaluated on the independent test set.<br><br>
Use these results together with the baseline notebook to document performance improvements and support later analysis and reporting.
"""

display(HTML(f"""
<div style="
    padding: 15px;
    border: 2px solid #4CAF50;
    background-color: #E8F5E9;
    border-radius: 8px;
    font-size: 16px;
">
{message}
</div>
"""))

